# Data Collection and Preparation

## Datasets Used

This notebook constructs a quarterly country-level panel dataset for EU-27 countries, combining sources that capture housing market dynamics from multiple angles: demand, supply, macroeconomic conditions, financing costs, and political environment.

| Dataset | Source | Frequency | Role |
|---|---|---|---|
| House Price Index (HPI) | Eurostat | Quarterly | Target variable |
| HICP Inflation | Eurostat | Monthly → Quarterly | Demand-side cost pressure |
| GDP Index | Eurostat | Quarterly | Economic activity |
| Unemployment Rate | Eurostat | Quarterly | Labour market conditions |
| Building Permits Index | Eurostat | Quarterly | Housing supply proxy |
| Old-age Dependency Ratio | Eurostat | Annual → Quarterly | Structural demographics |
| Population Density | Eurostat | Annual → Quarterly | Structural demographics |
| MIR Mortgage Rates | ECB API | Monthly → Quarterly | Cost of borrowing (eurozone only) |
| Long-term Interest Rates | Eurostat | Quarterly | Sovereign yields (all EU-27) |
| Manifesto Project (MARPOR) | MARPOR | Per election → Quarterly | Parliament ideology score |

## Key Processing Decisions

**Frequency alignment.** Monthly series (HICP, MIR) are aggregated to quarterly by simple average. Annual series (demographics) are expanded by repeating each year's value across all four quarters, appropriate for slow-moving structural variables.

**Missing data strategy.** Core time-series variables (growth rates, target) must be non-null — rows missing these are dropped. Slow-moving variables (unemployment, mortgage rates, long-term rates, ideology score) are forward-filled within each country. Mortgage rates are structurally missing for non-eurozone members; this is retained as informative missingness rather than imputed.

**Feature engineering.** Quarter-on-quarter percentage growth rates are computed for HPI, HICP, GDP, and building permits. The target variable is next-quarter HPI growth (shift of -1). All core predictors include two lags (t-1, t-2) to capture temporal dependencies and market inertia.

**Parliament ideology.** The MARPOR  score (right-left, ~-100 to +100) is aggregated per election as a seat-share-weighted average across all parties, then carried forward between elections. This reflects the full legislative balance of power rather than the governing coalition alone, as government membership is not available in the standard MARPOR release.


In [1]:
pip install pandas numpy matplotlib seaborn scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from io import StringIO
import requests

# 1. LOAD DATA
base_path = "./data"
#base_path = r"C:\Users\Giulia\Documents\GitHub\european-house-market-forecasting\data"

hpi     = pd.read_csv(f"{base_path}/house_price_index.csv", low_memory=False)
hicp    = pd.read_csv(f"{base_path}/Harmonised index of consumer prices (HICP).csv", low_memory=False)
unemp   = pd.read_csv(f"{base_path}/unemployment.csv", low_memory=False)
gdp     = pd.read_csv(f"{base_path}/Gross domestic product (GDP) and main components (output, expenditure and income).csv", low_memory=False)
permits = pd.read_csv(f"{base_path}/building_permits.csv", low_memory=False)
old_age = pd.read_csv(f"{base_path}/Old-age-dependency ratio.csv", low_memory=False)
popdens = pd.read_csv(f"{base_path}/Population density.csv", low_memory=False)
lt_raw  = pd.read_csv(f"{base_path}/long_term_rates.csv", low_memory=False)
mpd_raw = pd.read_csv(f"{base_path}/mpd_data.csv", low_memory=False)

# MIR data fetched from ECB API (eurozone members only)
eurozone_mir = ["AT","BE","CY","DE","EE","ES","FI","FR","GR","HR","IE",
                "IT","LT","LU","LV","MT","NL","PT","SI","SK"]
dfs = []
for cc in eurozone_mir:
    url = (
        f"https://data-api.ecb.europa.eu/service/data/MIR/"
        f"M.{cc}.B.A2C.AM.R.A.2250.EUR.N"
        f"?format=csvdata&startPeriod=2005-01"
    )
    r = requests.get(url)
    if r.status_code == 200 and len(r.text) > 100:
        df = pd.read_csv(StringIO(r.text))
        df["country_code"] = cc
        dfs.append(df)
mir = pd.concat(dfs, ignore_index=True)

# 2. EU-27 COUNTRY MAP
eu27 = {
    "Austria":"AT","Belgium":"BE","Bulgaria":"BG","Croatia":"HR","Cyprus":"CY",
    "Czechia":"CZ","Denmark":"DK","Estonia":"EE","Finland":"FI","France":"FR",
    "Germany":"DE","Greece":"EL","Hungary":"HU","Ireland":"IE","Italy":"IT",
    "Latvia":"LV","Lithuania":"LT","Luxembourg":"LU","Malta":"MT","Netherlands":"NL",
    "Poland":"PL","Portugal":"PT","Romania":"RO","Slovakia":"SK","Slovenia":"SI",
    "Spain":"ES","Sweden":"SE"
}
eu27_code_to_name = {v: k for k, v in eu27.items()}

def keep_eu27_by_name(df, country_col="geo"):
    df = df.copy()
    df = df[df[country_col].isin(eu27.keys())].copy()
    df["country_code"] = df[country_col].map(eu27)
    df["country"] = df[country_col]
    return df

# 3. HOUSE PRICE INDEX
hpi = hpi[hpi["unit"] == "I15_Q"].copy()
hpi = hpi[hpi["geo"].isin(eu27_code_to_name.keys())]
hpi = hpi.rename(columns={
    "geo": "country_code",
    "Geopolitical entity (reporting)": "country",
    "TIME_PERIOD": "quarter",
    "OBS_VALUE": "hpi"
})
hpi = hpi[["country_code","country","quarter","hpi"]].drop_duplicates()

# 4. HICP → QUARTERLY
hicp = hicp[(hicp["coicop18"]=="Total") & (hicp["unit"]=="Index, 2015=100")].copy()
hicp = keep_eu27_by_name(hicp)
hicp["date"] = pd.to_datetime(hicp["TIME_PERIOD"], format="%Y-%m")
hicp["quarter"] = hicp["date"].dt.to_period("Q").astype(str).str.replace("Q","-Q")
hicp_q = hicp.groupby(["country_code","country","quarter"], as_index=False)["OBS_VALUE"].mean()
hicp_q = hicp_q.rename(columns={"OBS_VALUE":"hicp_q"})

# 5. UNEMPLOYMENT
unemp = keep_eu27_by_name(unemp)
unemp = unemp[
    (unemp["sex"]=="Total") &
    (unemp["age"]=="From 15 to 74 years") &
    (unemp["unit"]=="Percentage of population in the labour force") &
    (unemp["s_adj"]=="Seasonally adjusted data, not calendar adjusted data")
]
unemp = unemp.rename(columns={"TIME_PERIOD":"quarter","OBS_VALUE":"unemployment_rate"})
unemp = unemp[["country_code","country","quarter","unemployment_rate"]]

# 6. GDP
gdp = keep_eu27_by_name(gdp)
gdp = gdp[
    (gdp["na_item"]=="Gross domestic product at market prices") &
    (gdp["s_adj"]=="Seasonally and calendar adjusted data") &
    (gdp["unit"]=="Chain linked volumes, index 2005=100")
]
gdp = gdp.rename(columns={"TIME_PERIOD":"quarter","OBS_VALUE":"gdp_index"})
gdp = gdp[["country_code","country","quarter","gdp_index"]]

# 7. BUILDING PERMITS
permits = keep_eu27_by_name(permits)
permits = permits[
    (permits["indic_bt"]=="Building permits - number of dwellings") &
    (permits["cpa2_1"]=="Residential buildings, except residences for communities") &
    (permits["unit"]=="Index, 2021=100")
]
permits = permits.rename(columns={"TIME_PERIOD":"quarter","OBS_VALUE":"building_permits_index"})
permits = permits[["country_code","country","quarter","building_permits_index"]]

# 8. ANNUAL → QUARTERLY (demographic variables)
def annual_to_quarter(df, value_col):
    df = keep_eu27_by_name(df)
    df = df.rename(columns={"TIME_PERIOD":"year","OBS_VALUE":value_col})
    df["year"] = df["year"].astype(int)
    out = []
    for q in ["Q1","Q2","Q3","Q4"]:
        tmp = df.copy()
        tmp["quarter"] = tmp["year"].astype(str)+"-"+q
        out.append(tmp[["country_code","country","quarter",value_col]])
    return pd.concat(out)

old_age_q = annual_to_quarter(old_age, "old_age_dependency_ratio")
popdens_q = annual_to_quarter(popdens, "population_density")

# 9. MIR — ECB Mortgage Interest Rates (eurozone members only)
mir["OBS_VALUE"] = pd.to_numeric(
    mir["OBS_VALUE"].astype(str).str.replace(",",".",regex=False).str.strip(),
    errors="coerce"
)
mir["date"] = pd.to_datetime(mir["TIME_PERIOD"], format="%Y-%m")
mir["quarter"] = mir["date"].dt.to_period("Q").astype(str).str.replace("Q","-Q")
mir_q = (
    mir.groupby(["country_code","quarter"], as_index=False)["OBS_VALUE"]
    .mean()
    .rename(columns={"OBS_VALUE":"mortgage_rate_pct"})
)
mir_q = mir_q[mir_q["country_code"].isin(eu27.values())].copy()
mir_q = mir_q.sort_values(["country_code","quarter"])
mir_q["mortgage_rate_pct"] = (
    mir_q.groupby("country_code")["mortgage_rate_pct"]
    .transform(lambda x: x.ffill())
)

# 10. LONG-TERM INTEREST RATES — Eurostat IRT_LT_MCBY_Q
lt_raw = lt_raw.rename(columns={
    "geo":"country","TIME_PERIOD":"quarter","OBS_VALUE":"lt_interest_rate"
})
lt_raw = lt_raw[lt_raw["country"].isin(eu27.keys())].copy()
lt_raw["country_code"] = lt_raw["country"].map(eu27)
lt_rates_q = (
    lt_raw[["country_code","quarter","lt_interest_rate"]]
    .drop_duplicates(subset=["country_code","quarter"])
    .sort_values(["country_code","quarter"])
)
lt_rates_q["lt_interest_rate"] = (
    lt_rates_q.groupby("country_code")["lt_interest_rate"]
    .transform(lambda x: x.ffill())
)

# 11. MANIFESTO PROJECT (MPD) — Parliament-weighted ideology score
mpd_country_map = {
    "Austria":"AT","Belgium":"BE","Bulgaria":"BG","Croatia":"HR",
    "Cyprus":"CY","Czech Republic":"CZ","Czechia":"CZ","Denmark":"DK",
    "Estonia":"EE","Finland":"FI","France":"FR","Germany":"DE",
    "Greece":"EL","Hungary":"HU","Ireland":"IE","Italy":"IT",
    "Latvia":"LV","Lithuania":"LT","Luxembourg":"LU","Malta":"MT",
    "Netherlands":"NL","Poland":"PL","Portugal":"PT","Romania":"RO",
    "Slovakia":"SK","Slovenia":"SI","Spain":"ES","Sweden":"SE"
}

mpd_raw["country_code"] = mpd_raw["countryname"].map(mpd_country_map)
mpd_eu = mpd_raw[mpd_raw["country_code"].notna()].copy()

for col in ["absseat","totseats","rile"]:
    mpd_eu[col] = pd.to_numeric(
        mpd_eu[col].astype(str).str.replace(",",".",regex=False).str.strip(),
        errors="coerce"
    )

mpd_eu = mpd_eu.dropna(subset=["rile","absseat","totseats"])
mpd_eu = mpd_eu[mpd_eu["totseats"] > 0].copy()

mpd_eu["election_date"] = pd.to_datetime(
    mpd_eu["date"].astype(str).str.zfill(6), format="%Y%m"
)
mpd_eu["election_quarter"] = (
    mpd_eu["election_date"].dt.to_period("Q").astype(str).str.replace("Q","-Q")
)
mpd_eu["seat_share"] = mpd_eu["absseat"] / mpd_eu["totseats"]

mpd_election = (
    mpd_eu.groupby(["country_code","election_quarter"])
    .apply(
        lambda g: np.average(g["rile"], weights=g["seat_share"]),
        include_groups=False
    )
    .reset_index()
    .rename(columns={0:"parliament_rile"})
)

# 12. MERGE
master = hpi.copy()

merge_dfs = [hicp_q, unemp, gdp, permits, old_age_q, popdens_q, mir_q, lt_rates_q]
for df in merge_dfs:
    df_clean = df.drop(columns=["country"], errors="ignore")
    master = master.merge(df_clean, on=["country_code","quarter"], how="left")

# 13. TIME FEATURES
master["year"] = master["quarter"].str[:4].astype(int)
master["quarter_num"] = master["quarter"].str[-1].astype(int)

# 14. EUROZONE FLAGS
current_eurozone = {"AT","BE","BG","HR","CY","EE","FI","FR","DE","EL","IE","IT",
                    "LV","LT","LU","MT","NL","PT","SK","SI","ES"}
master["eurozone_member_current"] = master["country_code"].isin(current_eurozone).astype(int)

adoption_year = {
    "AT":1999,"BE":1999,"BG":2026,"HR":2023,"CY":2008,"EE":2011,"FI":1999,
    "FR":1999,"DE":1999,"EL":2001,"IE":1999,"IT":1999,"LV":2014,"LT":2015,
    "LU":1999,"MT":2008,"NL":1999,"PT":1999,"SK":2009,"SI":2007,"ES":1999
}
master["eurozone_member_timevarying"] = master.apply(
    lambda r: int(r["country_code"] in adoption_year and r["year"] >= adoption_year[r["country_code"]]),
    axis=1
)

# 15. SORT + GROWTH RATES + TARGET
master = master.sort_values(["country_code","quarter"])

master["hpi_qoq_pct"]       = master.groupby("country_code")["hpi"].pct_change()*100
master["hicp_qoq_pct"]      = master.groupby("country_code")["hicp_q"].pct_change()*100
master["gdp_qoq_pct"]       = master.groupby("country_code")["gdp_index"].pct_change()*100
master["permits_qoq_pct"]   = master.groupby("country_code")["building_permits_index"].pct_change()*100
master["target_hpi_next_q"] = master.groupby("country_code")["hpi_qoq_pct"].shift(-1)

# 16. FORWARD FILL SLOW-MOVING VARIABLES
for col in ["old_age_dependency_ratio","population_density",
            "unemployment_rate","mortgage_rate_pct","lt_interest_rate"]:
    master[col] = master.groupby("country_code")[col].transform(lambda x: x.ffill())

# 17. MPD — expand elections to quarterly panel and forward-fill
all_quarters = (
    master[["country_code","quarter"]]
    .drop_duplicates()
    .sort_values(["country_code","quarter"])
)

mpd_q = all_quarters.merge(
    mpd_election,
    left_on=["country_code","quarter"],
    right_on=["country_code","election_quarter"],
    how="left"
).drop(columns=["election_quarter"], errors="ignore")

mpd_q["parliament_rile"] = (
    mpd_q.groupby("country_code")["parliament_rile"]
    .transform(lambda x: x.ffill())
)
mpd_q = mpd_q[["country_code","quarter","parliament_rile"]]

# Merge parliament_rile into master
master = master.merge(mpd_q, on=["country_code","quarter"], how="left")

# 18. BUILD MODEL DATASET
required_cols = [
    "hpi_qoq_pct","hicp_qoq_pct","gdp_qoq_pct",
    "permits_qoq_pct","target_hpi_next_q"
]
model_data = master.dropna(subset=required_cols).copy()

# 19. ADD LAGS
lag_cols = [
    "hpi_qoq_pct","hicp_qoq_pct","gdp_qoq_pct","permits_qoq_pct",
    "unemployment_rate","mortgage_rate_pct","lt_interest_rate","parliament_rile"
]
for col in lag_cols:
    model_data[f"{col}_lag1"] = model_data.groupby("country_code")[col].shift(1)
    model_data[f"{col}_lag2"] = model_data.groupby("country_code")[col].shift(2)

# Drop rows where core lag columns are missing (first 2 observations per country)
core_lag_required = [
    f"{c}_lag1" for c in
    ["hpi_qoq_pct","hicp_qoq_pct","gdp_qoq_pct","permits_qoq_pct","unemployment_rate"]
]
model_data = model_data.dropna(subset=core_lag_required)

# 20. UNEMPLOYMENT QOQ CHANGE
model_data["unemployment_qoq_change"] = (
    model_data.groupby("country_code")["unemployment_rate"].diff()
)

# 21. FINAL CHECKS
print("MASTER:", master.shape)
print("MODEL:", model_data.shape)
print("\nColumn list:")
print(model_data.columns.tolist())
print("\nMissing values (columns with any NaN):")
missing = model_data.isna().sum()
print(missing[missing > 0])
print("\nInterest rate & ideology coverage (non-null counts per country):")
print(model_data.groupby("country_code")[
    ["mortgage_rate_pct","lt_interest_rate","parliament_rile"]
].count())

C:\Users\Giulia\AppData\Local\Temp\ipykernel_24956\3843532653.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mpd_raw["country_code"] = mpd_raw["countryname"].map(mpd_country_map)


MASTER: (2007, 22)
MODEL: (1704, 39)

Column list:
['country_code', 'country', 'quarter', 'hpi', 'hicp_q', 'unemployment_rate', 'gdp_index', 'building_permits_index', 'old_age_dependency_ratio', 'population_density', 'mortgage_rate_pct', 'lt_interest_rate', 'year', 'quarter_num', 'eurozone_member_current', 'eurozone_member_timevarying', 'hpi_qoq_pct', 'hicp_qoq_pct', 'gdp_qoq_pct', 'permits_qoq_pct', 'target_hpi_next_q', 'parliament_rile', 'hpi_qoq_pct_lag1', 'hpi_qoq_pct_lag2', 'hicp_qoq_pct_lag1', 'hicp_qoq_pct_lag2', 'gdp_qoq_pct_lag1', 'gdp_qoq_pct_lag2', 'permits_qoq_pct_lag1', 'permits_qoq_pct_lag2', 'unemployment_rate_lag1', 'unemployment_rate_lag2', 'mortgage_rate_pct_lag1', 'mortgage_rate_pct_lag2', 'lt_interest_rate_lag1', 'lt_interest_rate_lag2', 'parliament_rile_lag1', 'parliament_rile_lag2', 'unemployment_qoq_change']

Missing values (columns with any NaN):
old_age_dependency_ratio    508
population_density          404
mortgage_rate_pct           561
lt_interest_rate     

In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Export final model-ready dataset
#model_data.to_excel("./model_data.xlsx", index=False)
model_data.to_excel(r"C:\Users\Giulia\Documents\GitHub\european-house-market-forecasting\model_data.xlsx", index=False)
print("Exported model_data.xlsx")

Exported model_data.xlsx
